In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('kc_house_data.csv')
df = df.drop(columns=['id', 'date'])

Y = df[['price']].values
X_df = df.drop(columns=['price'])
X_df.insert(0, 'intercepto', 1)
X = X_df.values

print("Dimensión de X:", X.shape)
print("Dimensión de Y:", Y.shape)

Dimensión de X: (21613, 19)
Dimensión de Y: (21613, 1)


In [2]:
XT_X = np.matmul(np.matrix.transpose(X), X)
XT_X_inv = np.linalg.inv(XT_X)
XT_Y = np.matmul(np.matrix.transpose(X), Y)
betas = np.matmul(XT_X_inv, XT_Y)

print(betas)

[[-2.86822882e+08]
 [ 8.05348558e+07]
 [ 4.63488899e+06]
 [ 3.61267200e+06]
 [ 1.05026420e+01]
 [-2.41494614e+07]
 [ 2.20154913e+08]
 [-1.78060845e+07]
 [-1.82140884e+06]
 [ 8.62691304e+06]
 [-3.63724800e+06]
 [-3.62905600e+06]
 [-2.62022353e+03]
 [ 1.98125814e+01]
 [-5.82419623e+02]
 [ 6.02748247e+05]
 [-2.14729567e+05]
 [ 2.16813891e+01]
 [-3.82641965e-01]]


In [3]:
Y_pred = np.matmul(X, betas)
Resid = Y - Y_pred
RSS = np.matmul(np.matrix.transpose(Resid), Resid).item()

grados_libertad = len(Y) - X.shape[1]
s_cuad = RSS / grados_libertad

import math
s = math.sqrt(s_cuad)
print(s)

80767236.30727403


In [4]:
import scipy.stats

t_critico = abs(scipy.stats.t.ppf(q=0.025, df=grados_libertad))
result_t = []
columnas = X_df.columns

print(f"Valor t-crítico de corte: {t_critico:.4f}\n")

for i in range(len(columnas)):
    beta_i = betas[i, 0] if betas.ndim > 1 else betas[i]
    se = s * math.sqrt(XT_X_inv[i, i])
    t = beta_i / se
    result_t.append(t)
    
    estado = "SIGNIFICATIVA" if abs(t) > t_critico else "NO significativa"
    print(f"{columnas[i]:<15} -> {estado}")

Valor t-crítico de corte: 1.9601

intercepto      -> NO significativa
bedrooms        -> SIGNIFICATIVA
bathrooms       -> SIGNIFICATIVA


ValueError: math domain error

In [6]:
import pandas as pd
import numpy as np
import math

# Recargamos la base desde cero para limpiar la memoria de Jupyter
df_clean = pd.read_csv('kc_house_data.csv')

# Definimos nuestra variable objetivo
Y = df_clean[['price']].values

# Armamos la matriz X quitando las que no sirven (id, date) y las redundantes (sqft_above, sqft_basement)
X_df = df_clean.drop(columns=['id', 'date', 'price', 'sqft_above', 'sqft_basement'])
X_df.insert(0, 'intercepto', 1)
X = X_df.values

# Recalculamos la matemática (Betas)
XT_X = np.matmul(np.matrix.transpose(X), X)
XT_X_inv = np.linalg.inv(XT_X)
XT_Y = np.matmul(np.matrix.transpose(X), Y)
betas = np.matmul(XT_X_inv, XT_Y)

# Calculamos el error real
Y_pred = np.matmul(X, betas)
Resid = Y - Y_pred
RSS = np.matmul(np.matrix.transpose(Resid), Resid).item()
grados_libertad = len(Y) - X.shape[1]
s = math.sqrt(RSS / grados_libertad)

print("Error promedio del modelo reajustado (s):", round(s, 2))

Error promedio del modelo reajustado (s): 201480.38


In [7]:
import scipy.stats
import math

t_critico = abs(scipy.stats.t.ppf(q=0.025, df=grados_libertad))
print(f"Valor límite (t-crítico): {t_critico:.4f}\n")

columnas = X_df.columns
for i in range(len(columnas)):
    beta_i = betas[i, 0] if betas.ndim > 1 else betas[i]
    se = s * math.sqrt(abs(XT_X_inv[i, i]))
    t = beta_i / se
    
    estado = "SIGNIFICATIVA" if abs(t) > t_critico else "NO significativa"
    print(f"{columnas[i]:<15} -> {estado} (t = {t:.2f})")

Valor límite (t-crítico): 1.9601

intercepto      -> SIGNIFICATIVA (t = 3.31)
bedrooms        -> SIGNIFICATIVA (t = -18.95)
bathrooms       -> SIGNIFICATIVA (t = 11.67)
sqft_living     -> SIGNIFICATIVA (t = 50.97)
sqft_lot        -> SIGNIFICATIVA (t = 2.90)
floors          -> SIGNIFICATIVA (t = 5.59)
waterfront      -> SIGNIFICATIVA (t = 33.82)
view            -> SIGNIFICATIVA (t = 23.83)
condition       -> SIGNIFICATIVA (t = 10.67)
grade           -> SIGNIFICATIVA (t = 45.99)
yr_built        -> SIGNIFICATIVA (t = -36.13)
yr_renovated    -> SIGNIFICATIVA (t = 5.35)
zipcode         -> SIGNIFICATIVA (t = -17.80)
lat             -> SIGNIFICATIVA (t = 55.63)
long            -> SIGNIFICATIVA (t = -15.33)
sqft_living15   -> SIGNIFICATIVA (t = 7.38)
sqft_lot15      -> SIGNIFICATIVA (t = -5.10)


In [8]:
print("--- CRITERIO 2: VALORES P (p-values) ---")
for i in range(len(columnas)):
    beta_i = betas[i, 0] if betas.ndim > 1 else betas[i]
    se = s * math.sqrt(abs(XT_X_inv[i, i]))
    t = beta_i / se
    
    # Cálculo del valor p a dos colas
    p_value = 2 * (1 - scipy.stats.t.cdf(abs(t), df=grados_libertad))
    
    estado = "SIGNIFICATIVA" if p_value < 0.05 else "NO significativa"
    
    # Formateo para evitar notación científica extrema en pantalla
    p_format = "< 0.0001" if p_value < 0.0001 else f"{p_value:.4f}"
    print(f"{columnas[i]:<15} -> {estado} (p-value: {p_format})")

--- CRITERIO 2: VALORES P (p-values) ---
intercepto      -> SIGNIFICATIVA (p-value: 0.0009)
bedrooms        -> SIGNIFICATIVA (p-value: < 0.0001)
bathrooms       -> SIGNIFICATIVA (p-value: < 0.0001)
sqft_living     -> SIGNIFICATIVA (p-value: < 0.0001)
sqft_lot        -> SIGNIFICATIVA (p-value: 0.0037)
floors          -> SIGNIFICATIVA (p-value: < 0.0001)
waterfront      -> SIGNIFICATIVA (p-value: < 0.0001)
view            -> SIGNIFICATIVA (p-value: < 0.0001)
condition       -> SIGNIFICATIVA (p-value: < 0.0001)
grade           -> SIGNIFICATIVA (p-value: < 0.0001)
yr_built        -> SIGNIFICATIVA (p-value: < 0.0001)
yr_renovated    -> SIGNIFICATIVA (p-value: < 0.0001)
zipcode         -> SIGNIFICATIVA (p-value: < 0.0001)
lat             -> SIGNIFICATIVA (p-value: < 0.0001)
long            -> SIGNIFICATIVA (p-value: < 0.0001)
sqft_living15   -> SIGNIFICATIVA (p-value: < 0.0001)
sqft_lot15      -> SIGNIFICATIVA (p-value: < 0.0001)


In [9]:
print("--- CRITERIO 3: INTERVALOS DE CONFIANZA ---")
for i in range(len(columnas)):
    beta_i = betas[i, 0] if betas.ndim > 1 else betas[i]
    se = s * math.sqrt(abs(XT_X_inv[i, i]))
    
    lim_inf = beta_i - (t_critico * se)
    lim_sup = beta_i + (t_critico * se)
    
    if lim_inf <= 0 <= lim_sup:
        estado = "NO significativa (Contiene 0)"
    else:
        estado = "SIGNIFICATIVA (No contiene 0)"
        
    print(f"{columnas[i]:<15} -> {estado} [{lim_inf:.2f}, {lim_sup:.2f}]")

--- CRITERIO 3: INTERVALOS DE CONFIANZA ---
intercepto      -> SIGNIFICATIVA (No contiene 0) [3925549.43, 15317282.88]
bedrooms        -> SIGNIFICATIVA (No contiene 0) [-39602.98, -32178.42]
bathrooms       -> SIGNIFICATIVA (No contiene 0) [31244.68, 43860.74]
sqft_living     -> SIGNIFICATIVA (No contiene 0) [163.84, 176.95]
sqft_lot        -> SIGNIFICATIVA (No contiene 0) [0.05, 0.23]
floors          -> SIGNIFICATIVA (No contiene 0) [11721.42, 24376.99]
waterfront      -> SIGNIFICATIVA (No contiene 0) [553420.79, 621508.59]
view            -> SIGNIFICATIVA (No contiene 0) [46217.55, 54502.35]
condition       -> SIGNIFICATIVA (No contiene 0) [20431.29, 29629.90]
grade           -> SIGNIFICATIVA (No contiene 0) [93915.27, 102276.82]
yr_built        -> SIGNIFICATIVA (No contiene 0) [-2770.49, -2485.36]
yr_renovated    -> SIGNIFICATIVA (No contiene 0) [12.41, 26.75]
zipcode         -> SIGNIFICATIVA (No contiene 0) [-652.43, -523.00]
lat             -> SIGNIFICATIVA (No contiene 0) [572922